# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [43]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import chi2_contingency
SEED=1

In [44]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [45]:
spaceship.dropna(inplace=True)

X_train_spaceship_cabin = spaceship.copy()

X_train_spaceship_cabin['Cabin'] = (
    X_train_spaceship_cabin['Cabin'].str.split('/').str[0]
)

X_train_spaceship_cabin['Cabin'].value_counts(dropna=False)
X_train_spaceship_cabin = X_train_spaceship_cabin.drop(columns=['PassengerId', 'Name'])
X = X_train_spaceship_cabin.drop(columns='Transported')
y = X_train_spaceship_cabin['Transported'].astype(int)

X = pd.get_dummies(X, dtype=int)
X.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_False,...,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_False,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,0,1,0,1,...,0,0,0,0,0,0,0,1,1,0
1,24.0,109.0,9.0,25.0,549.0,44.0,1,0,0,1,...,0,0,1,0,0,0,0,1,1,0
2,58.0,43.0,3576.0,0.0,6715.0,49.0,0,1,0,1,...,0,0,0,0,0,0,0,1,0,1
3,33.0,0.0,1283.0,371.0,3329.0,193.0,0,1,0,1,...,0,0,0,0,0,0,0,1,1,0
4,16.0,303.0,70.0,151.0,565.0,2.0,1,0,0,1,...,0,0,1,0,0,0,0,1,1,0


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [46]:
# Step 2: Train / Test Split
# ---------------------------
# Train / Test Split (without stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED)
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')

X_train shape: (5284, 24)
X_test shape:  (1322, 24)


In [47]:
# XGBoost NEEDS TUNING to show its power:
from xgboost import XGBRegressor, XGBClassifier
xgb_clf = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=10,
    random_state=SEED,
    eval_metric='logloss')

xgb_clf.fit(X_train, y_train)
print("Model Trained!")

Model Trained!


- Evaluate your model

In [48]:
#your code here# Step 5 - Results
# -------------------
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred_train = xgb_clf.predict(X_train)
y_pred_test = xgb_clf.predict(X_test)

acc_train = accuracy_score(y_train, y_pred_train)
acc_test = accuracy_score(y_test, y_pred_test)

print(f'Accuracy (Train): {acc_train * 100:.2f}%')
print(f'Accuracy (Test): {acc_test * 100:.2f}%')
print()
print('Classification Report (Test):')
print(classification_report(y_test, y_pred_test))

Accuracy (Train): 87.40%
Accuracy (Test): 78.97%

Classification Report (Test):
              precision    recall  f1-score   support

           0       0.79      0.77      0.78       648
           1       0.79      0.80      0.80       674

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322



**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [49]:
# The grid: the same 5 hyperparameters we actually tuned by hand in Section 4
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


grid = {
    'n_estimators': [200, 300, 500, 600],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_lambda': [1, 3, 10, 20]}
xgb_clf = XGBClassifier(random_state=SEED)

# cv=4 --> same Cross Validation with K=4
# n_iter=10 --> instead of trying all combinations, only 10 random ones are sampled
model = RandomizedSearchCV(estimator=xgb_clf,
                           param_distributions=grid,
                           n_iter=10,
                           cv=4,
                           random_state=SEED,      # So all the random iterations are the same
                           n_jobs=-1)              # n_jobs=-1 --> allows multicore CPU processing
model.fit(X_train, y_train)

print("Model Trained!")

Model Trained!


- Run Grid Search

- Evaluate your model

In [51]:




from sklearn.metrics import accuracy_score, classification_report

print("Best hyperparameters found:")
print(model.best_params_)

best_xgb_clf = model.best_estimator_

y_pred_train = best_xgb_clf.predict(X_train)
y_pred_test = best_xgb_clf.predict(X_test)

acc_train = accuracy_score(y_train, y_pred_train)
acc_test = accuracy_score(y_test, y_pred_test)

print(f"Accuracy (Train): {acc_train * 100:.2f}%")
print(f"Accuracy (Test): {acc_test * 100:.2f}%")
print()
print("Classification Report (Test):")
print(classification_report(y_test, y_pred_test))

Best hyperparameters found:
{'subsample': 1.0, 'reg_lambda': 1, 'n_estimators': 500, 'learning_rate': 0.01, 'colsample_bytree': 1.0}
Accuracy (Train): 83.74%
Accuracy (Test): 80.03%

Classification Report (Test):
              precision    recall  f1-score   support

           0       0.81      0.77      0.79       648
           1       0.79      0.83      0.81       674

    accuracy                           0.80      1322
   macro avg       0.80      0.80      0.80      1322
weighted avg       0.80      0.80      0.80      1322



In [62]:
from sklearn.model_selection import train_test_split, GridSearchCV



from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

grid = {
    "n_estimators": [600, 800, 900],
    "learning_rate": [0.01],
    "subsample": [1.0],
    "colsample_bytree": [0.5, 1.0],
    "reg_lambda": [1],
}

xgb_clf = XGBClassifier(
    random_state=SEED,
    eval_metric="logloss"
)

model = GridSearchCV(
    estimator=xgb_clf,
    param_grid=grid,
    scoring="accuracy",
    cv=4,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Model Trained!")
print("Best parameters:", model.best_params_)
print("Best CV accuracy:", model.best_score_)

Model Trained!
Best parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.01, 'n_estimators': 600, 'reg_lambda': 1, 'subsample': 1.0}
Best CV accuracy: 0.8046934140802422


In [61]:
print("Best hyperparameters found:")
print(model.best_params_)

best_xgb_clf = model.best_estimator_

y_pred_train = best_xgb_clf.predict(X_train)
y_pred_test = best_xgb_clf.predict(X_test)

acc_train = accuracy_score(y_train, y_pred_train)
acc_test = accuracy_score(y_test, y_pred_test)

print(f"Accuracy (Train): {acc_train * 100:.2f}%")
print(f"Accuracy (Test): {acc_test * 100:.2f}%")
print()
print("Classification Report (Test):")
print(classification_report(y_test, y_pred_test))

Best hyperparameters found:
{'colsample_bytree': 1.0, 'learning_rate': 0.01, 'n_estimators': 600, 'reg_lambda': 1, 'subsample': 1.0}
Accuracy (Train): 83.82%
Accuracy (Test): 79.80%

Classification Report (Test):
              precision    recall  f1-score   support

           0       0.81      0.76      0.79       648
           1       0.78      0.83      0.81       674

    accuracy                           0.80      1322
   macro avg       0.80      0.80      0.80      1322
weighted avg       0.80      0.80      0.80      1322

